In [1]:
import pandas as pd
from SynFileUtils import SynFileReader

In [2]:
mirna_path = '/mnt/raidbio2/extproj/projekte/textmining/mirnaTextmining/mirFormTM/results/mirna.hits'
taxon_path = '/mnt/raidbio2/extproj/projekte/textmining/mirnaTextmining/mirClassification/taxon_hits_20260529_182015/taxon_hits_20260529_182015.hits'
mirna_prefixes = '/mnt/raidbio2/extproj/projekte/textmining/mirnaTextmining/mirClassification/data/mirnas_prefix.tsv'
taxon_syns = '/mnt/raidbio2/extstud/studtemp/mitsopoulos/synonyms/ncbi_taxonomy.syn'

mirna_hits = pd.read_csv(mirna_path, sep='\t', quoting=3)
taxon_hits = pd.read_csv(taxon_path, sep ='\t', quoting=3)
mirna_prefixes = pd.read_csv(mirna_prefixes, sep='\t')
mirna_prefixes.head()

,prefix,organism,id
0,nve,Nematostella vectensis,NCBITaxon:45351
1,hma,Hydra magnipapillata,NCBITaxon:6085
2,sko,Saccoglossus kowalevskii,NCBITaxon:10224
3,spu,Strongylocentrotus purpuratus,NCBITaxon:7668
4,cin,Ciona intestinalis,NCBITaxon:7719


In [3]:
columns = [
    "sentence_id",
    "synonym_id",
    "matched_text",
    "start_position",
    "hit_length",
    "synonym",
    "prefix",
    "suffix",
]

mirna_hits.drop(mirna_hits.columns[8], axis=1, inplace=True)
mirna_hits.columns = columns
taxon_hits.columns = columns

mirna_hits['sentence_id'] = mirna_hits['sentence_id'].str.split(':').str[1]
taxon_hits['sentence_id'] = taxon_hits['sentence_id'].str.split(':').str[1]

In [4]:
# Extract article id
mirna_hits['article_id'] = mirna_hits['sentence_id'].str.split('.').str[0]
taxon_hits['article_id'] = taxon_hits['sentence_id'].str.split('.').str[0]

In [5]:
with SynFileReader(taxon_syns) as reader:
    line_numbers = taxon_hits["synonym_id"].str.split(":").str[1].astype(int)
    taxon_hits["taxon_id"] = line_numbers.apply(reader.extract_id)

taxon_hits.head()

,sentence_id,synonym_id,matched_text,start_position,hit_length,synonym,prefix,suffix,article_id,taxon_id
0,PMC12631457.8.12,0:1208011,clinical samples,1589,16,clinical samples,NaN,.,PMC12631457,NCBITaxon:226901
1,PMC12631457.9.12,0:1208011,clinical samples (,218,18,clinical samples,NaN,NaN,PMC12631457,NCBITaxon:226901
2,PMC12631461.5.32,0:10704,rats (,160,6,rats,NaN,NaN,PMC12631461,NCBITaxon:10114
3,PMC12631461.5.50,0:10704,rats (,142,6,rats,NaN,NaN,PMC12631461,NCBITaxon:10114
4,PMC12631461.5.64,0:10923,rat,110,3,rat,NaN,NaN,PMC12631461,NCBITaxon:10116


In [19]:
taxon_hits['taxon_id'].isna().sum() == 0

True

In [20]:
valid_tax_ids = set(mirna_prefixes['id'])
valid_taxons = taxon_hits['taxon_id'].isin(valid_tax_ids)
taxon_hits_filtered = taxon_hits[valid_taxons].reset_index(drop=True)
taxon_hits_filtered.head()

,sentence_id,synonym_id,matched_text,start_position,hit_length,synonym,prefix,suffix,article_id,taxon_id
0,PMC12631461.5.64,0:10923,rat,110,3,rat,NaN,NaN,PMC12631461,NCBITaxon:10116
1,PMC12631461.5.91,0:10923,rat,154,3,rat,NaN,NaN,PMC12631461,NCBITaxon:10116
2,PMC12631461.6.16,0:10923,rat,205,3,rat,NaN,NaN,PMC12631461,NCBITaxon:10116
3,PMC12631461.6.113,0:10923,rat,155,3,rat,NaN,NaN,PMC12631461,NCBITaxon:10116
4,PMC12631531.3.5,0:2270751,rice,21,4,rice,NaN,NaN,PMC12631531,NCBITaxon:4530


In [25]:
# Filtering

# Remove any special characters that are not hyphens
mirna_hits["prefix"] = (
    mirna_hits["prefix"]
    .str.replace(r"[^a-zA-Z0-9-]", "", regex=True)
    .str.strip()
)

# Set everything to lower-case
mirna_hits['prefix'] = mirna_hits['prefix'].str.lower()

# Load valid prefixes and remove any prefix that does not have a valid prefix.
valid_prefixes = set(mirna_prefixes['prefix'] + '-')
is_valid_prefix = mirna_hits["prefix"].fillna("").isin(valid_prefixes)
mirna_hits.loc[~is_valid_prefix, "prefix"] = None


In [34]:
joined = pd.merge(mirna_hits[['sentence_id', 'synonym', 'prefix', 'suffix']], taxon_hits_filtered[['sentence_id', 'matched_text', 'taxon_id']], how='inner', on='sentence_id')

In [ ]:
tax_id_to_prefix = dict(zip(mirna_prefixes['id'], mirna_prefixes['prefix'] + '-'))
joined['implied_prefix'] = joined['taxon_id'].map(tax_id_to_prefix)
mismatch = joined[joinned]

,sentence_id,synonym,prefix,suffix,matched_text,taxon_id,implied_prefix
0,PMC12100665.6.69,miR,NaN,397,rice (,NCBITaxon:4530,osa
1,PMC12100665.6.71,miR,NaN,397a,Brassica napus),NCBITaxon:3708,bna
2,PMC12101272.5.174,miR,NaN,-146a,Zebrafish,NCBITaxon:7955,dre
3,PMC12101272.5.174,miR,NaN,-146a,Zebrafish,NCBITaxon:7955,dre
4,PMC12101272.5.177,miR,NaN,-375,zebrafish,NCBITaxon:7955,dre


In [49]:
prefix_clusters = mirna_hits_filtered.groupby('article_id')['prefix'].apply(lambda x: set(x.dropna())).reset_index()
mask = prefix_clusters["prefix"].apply(len) >= 2
multi_prefix_count = mask.sum()

# 3. Print the results
print(f"Total number of unique articles: {len(prefix_clusters)}")
print(
    f"Articles with at least two unique prefixes: {multi_prefix_count}"
)

prefix_clusters[mask].tail()



Total number of unique articles: 42121
Articles with at least two unique prefixes: 4502


,article_id,prefix
42070,PMC9986394,"{hsa-, mmu-}"
42080,PMC9988957,"{mja-, pol-, dre-, bta-, fru-}"
42084,PMC9990295,"{mmu-, hsa-}"
42088,PMC9992195,"{cel-, osa-, hsa-}"
42107,PMC9996378,"{cel-, hsa-}"


In [ ]:
mirna_hits['combined'] = mirna_hits['prefix'].fillna('') + mirna_hits['matched_text'] + mirna_hits['suffix'].fillna('')
mirna_hits.head()

,sentence_id,synonym_id,matched_text,start_position,hit_length,synonym,prefix,suffix,article_id,combined
0,PMC12100665.2.11,0:0,miR,4,3,miR,Ath-,397,PMC12100665,Ath-miR397
1,PMC12100665.3.45,0:0,miR,4,3,miR,NaN,397,PMC12100665,miR397
2,PMC12100665.5.100,0:0,miR,125,3,miR,NaN,397,PMC12100665,miR397
3,PMC12100665.5.101,0:0,miR,18,3,miR,Ath-,397a,PMC12100665,Ath-miR397a
4,PMC12100665.5.102,0:0,miR,19,3,miR,Ath-,397b,PMC12100665,Ath-miR397b


In [27]:
result = mirna_hits.groupby('article_id')['combined'].apply(lambda x: set(x.dropna())).reset_index()
result.head()

,article_id,combined
0,10028644,"{LSY 4., LSY 4}"
1,10066355,{Euromir-95}
2,10177113,"{(Bantam Books,}"
3,10188023,{bantams}
4,10355369,{Bantam fowl}


In [28]:
result.to_csv('/mnt/raidbio2/extproj/projekte/textmining/mirnaTextmining/mirClassification/data/per_article_mirnas.tsv', sep='\t')